# Explore Eurostat population data.

In this example, you use Copilot for Fabric Data Engineering to load, transform, and save data in a lakehouse. Contoso Health, a multispecialty hospital network, wants to expand its services in the European Union (EU) and analyze projected population data. This example uses the [Eurostat](https://ec.europa.eu/eurostat/web/main/home) population projection dataset.

Source: EUROPOP2023 Population on January 1 by age, sex, and type of projection [[proj_23np](https://ec.europa.eu/eurostat/databrowser/product/view/proj_23np?category=proj.proj_23n)], last updated June 28, 2023.

Before you start, make sure you have:

- Access to a Microsoft Fabric tenant account with an active subscription.
- A workspace with Fabric capacity.
- A lakehouse attached to this notebook.

This solution notebook contains example code generated from the prompts in the lab. To generate code yourself, open the **Copilot** chat pane from the **Home** ribbon, submit each prompt, review any requested notebook changes, and select **Allow**. Review generated code before you run it, and select **Keep** after it runs successfully.

## Load data

The following code downloads the Eurostat data and saves it to the attached lakehouse.

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

import os
import urllib.request

# Define the URL and the destination folder in the lakehouse
url = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/proj_23np$defaultview/?format=TSV"
lakehouse_path = "/lakehouse/default/Files/temp"
file_path = f"{lakehouse_path}/proj_23np.tsv"

# Create the folder if it doesn't exist
if not os.path.exists(lakehouse_path):
    os.makedirs(lakehouse_path)

# Download the file and save it to the specified location
urllib.request.urlretrieve(url, file_path)

print(f"File downloaded and saved to {file_path}")

Once we have the data downloaded to our lakehouse, we want to load it into a dataframe so we can look at the data and start transforming it.

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

# Define the file path
file_path = "Files/temp/proj_23np.tsv"

# Load the TSV into a Spark DataFrame
spark_df = spark.read.format("csv").option("header", "true").option("delimiter", "\t").load(file_path)

# Display the contents of the DataFrame
display(spark_df)

## Transform data: split fields

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

from pyspark.sql.functions import split, col

# Split the first field 'freq,projection,sex,age,unit,geo\TIME_PERIOD' into 6 separate fields
spark_df = spark_df.withColumn("freq", split(col("freq,projection,sex,age,unit,geo\\TIME_PERIOD"), ",").getItem(0)) \
                   .withColumn("projection", split(col("freq,projection,sex,age,unit,geo\\TIME_PERIOD"), ",").getItem(1)) \
                   .withColumn("sex", split(col("freq,projection,sex,age,unit,geo\\TIME_PERIOD"), ",").getItem(2)) \
                   .withColumn("age", split(col("freq,projection,sex,age,unit,geo\\TIME_PERIOD"), ",").getItem(3)) \
                   .withColumn("unit", split(col("freq,projection,sex,age,unit,geo\\TIME_PERIOD"), ",").getItem(4)) \
                   .withColumn("geo", split(col("freq,projection,sex,age,unit,geo\\TIME_PERIOD"), ",").getItem(5))

# Show the updated DataFrame
display(spark_df)

## Transform data: remove fields

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

# Remove the fields 'freq', 'age', and 'unit'
spark_df = spark_df.drop('freq', 'age', 'unit')

# Display the updated DataFrame
display(spark_df)

## Transform data: reposition fields

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

# Reorder the columns to place 'projection', 'sex', and 'geo' first
columns = ['projection', 'sex', 'geo'] + [col for col in spark_df.columns if col not in ['projection', 'sex', 'geo']]
spark_df = spark_df.select(columns)

# Display the reordered DataFrame
display(spark_df)

## Transform data: replace values

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

from pyspark.sql.functions import when

# Replace projection codes
spark_df = spark_df.withColumn("projection", 
                               when(spark_df["projection"] == "BSL", "Baseline projections")
                               .when(spark_df["projection"] == "LFRT", "Sensitivity test: lower fertility")
                               .when(spark_df["projection"] == "LMRT", "Sensitivity test: lower mortality")
                               .when(spark_df["projection"] == "HMIGR", "Sensitivity test: higher migration")
                               .when(spark_df["projection"] == "LMIGR", "Sensitivity test: lower migration")
                               .when(spark_df["projection"] == "NMIGR", "Sensitivity test: no migration")
                               .otherwise(spark_df["projection"]))

# Display the updated DataFrame
display(spark_df)

## Transform data: filter data

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

# Filter out 'geo' values 'EA20' and 'EU27_2020'
spark_df = spark_df.filter((spark_df['geo'] != 'EA20') & (spark_df['geo'] != 'EU27_2020'))

# Display the filtered DataFrame
display(spark_df)

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

# Filter out 'sex' values 'T'
spark_df = spark_df.filter(spark_df['sex'] != 'T')

# Display the filtered DataFrame
display(spark_df)

## Transform data: strip spaces

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

from pyspark.sql.functions import col

# Strip spaces from all field names
spark_df = spark_df.select([col(column).alias(column.strip()) for column in spark_df.columns])

# Display the updated DataFrame
display(spark_df)

## Transform data: data type conversion

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

from pyspark.sql.functions import col

# Convert the data type of all the year fields to integer
year_columns = [col(column).cast("int") for column in spark_df.columns if column.strip().isdigit()]
spark_df = spark_df.select(*spark_df.columns[:3], *year_columns)

# Display the updated DataFrame
display(spark_df)

## Save data

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

# Save the DataFrame as a new table named 'Population' in the default lakehouse
spark_df.write.format("delta").saveAsTable("Population")

## Ask questions

In [ ]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

import plotly.graph_objs as go

# Query to get the projected population trends for geo BE from 2022 to 2050
result = spark.sql(
    """
    SELECT `2022`, `2023`, `2025`, `2030`, `2035`,
           `2040`, `2045`, `2050`
    FROM Population
    WHERE geo = 'BE' AND projection = 'Baseline projections'
    """
)
df = result.toPandas()

# Extract data for the line chart
years = df.columns.tolist()
values = df.iloc[0].tolist()

# Create the plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=years, y=values, mode='lines+markers', name='Projected Population'))

# Update layout
fig.update_layout(
    title='Projected Population Trends for Geo BE (Belgium) from 2022 to 2050',
    xaxis_title='Year',
    yaxis_title='Population',
    template='plotly_dark'
)

# Display the plot
fig.show()
